# 1. Configuración de Entorno y Memoria
Configuramos variables de entorno para evitar la fragmentación de la VRAM y creamos una función para limpiar la basura de la GPU entre ejecuciones.


In [18]:
import os
# Evitar fragmentación de memoria en PyTorch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.tuner import Tuner
import torchvision
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset
import matplotlib.pyplot as plt
import random
import numpy as np

def cleanup():
    """Fuerza la recolección de basura y vacía la caché de CUDA"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


# 2. Carga de Datos (DataModule)
Separamos las transformaciones de CPU (lectura) y GPU (matemáticas pesadas). Usamos LightningDataModule para una gestión limpia.


In [19]:
# Transformaciones BASE (CPU) - Leer, redimensionar y pasar a Tensor [0, 1]
base_transforms = v2.Compose([
    v2.Grayscale(num_output_channels=1),
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# Transformaciones de Aumento (GPU)
gpu_train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomRotation(20),
    v2.Normalize(mean=[0.5], std=[0.5])
])

gpu_val_test_transforms = v2.Compose([
    v2.Normalize(mean=[0.5], std=[0.5])
])

class AnukaDataModule(L.LightningDataModule):
    def __init__(self, dataset_path, batch_size=64, num_workers=4):
        super().__init__()
        self.dataset_path = dataset_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.base_transforms = base_transforms

    def setup(self, stage=None):
        dataset = ImageFolder(root=self.dataset_path, transform=self.base_transforms)
        total_size = len(dataset)
        indices = list(range(total_size))
        train_indices, val_indices, test_indices = random_split(
            indices, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(42)
        )
        self.train_ds = Subset(dataset, train_indices)
        self.val_ds = Subset(dataset, val_indices)
        self.test_ds = Subset(dataset, test_indices)
        self.classes = dataset.classes

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

# Instanciamos el DataModule con un batch_size base (lo optimizaremos luego)
dm = AnukaDataModule(dataset_path='dataset/anuka1200', batch_size=32, num_workers=4)
dm.setup()
print(f"Classes: {dm.classes}, Train size: {len(dm.train_ds)}")

Classes: ['Tipo A: Kunzea', 'Tipo B: Lepto'], Train size: 1920


# 3. Definición del Módulo de Entrenamiento
Añadimos `on_after_batch_transfer` para ejecutar las transformaciones en la GPU.


In [20]:
class RedNeuronal(L.LightningModule):
    def __init__(self, model, is_cnn=False, lr=1e-3, train_transforms=None, val_transforms=None):
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'train_transforms', 'val_transforms'])
        self.model = model
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.is_cnn = is_cnn
        self.lr = lr
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms

    def on_after_batch_transfer(self, batch, dataloader_idx):
        # Mueve las transformaciones costosas a la GPU (CUDA)
        x, y = batch
        if self.trainer.training and self.train_transforms:
            x = self.train_transforms(x)
        elif self.val_transforms:
            x = self.val_transforms(x)
        return x, y

    def training_step(self, batch, batch_idx):
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_accuracy', accuracy, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):    
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('val_accuracy', accuracy, prog_bar=True)
        self.log('val_loss', loss, prog_bar=True)
        
    def test_step(self, batch, batch_idx):
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_accuracy', accuracy, prog_bar=True)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

# 4. Búsqueda de Batch Size Óptimo (Con techo de seguridad)


In [21]:
cleanup() # Aseguramos VRAM limpia

modelo_stress = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 28 * 28, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 1)
)

modelo_lig_stress = RedNeuronal(modelo_stress, is_cnn=True, 
                               train_transforms=gpu_train_transforms, 
                               val_transforms=gpu_val_test_transforms)

trainer_finder = L.Trainer(accelerator="gpu", devices=1)
tuner = Tuner(trainer_finder)

print("Buscando el Batch Size máximo...")
try:
    new_batch_size = tuner.scale_batch_size(modelo_lig_stress, 
                                            datamodule=dm,
                                            method="fit",
                                            mode="power",
                                            init_val=32, 
                                            max_trials=8)
    
    # TECHO DE SEGURIDAD: Máximo 256 para evitar OOM durante entrenamientos largos
    batch_size_seguro = min(new_batch_size, 64)
    dm.batch_size = batch_size_seguro
    print(f"Batch size bruto encontrado: {new_batch_size}. Batch size aplicado: {dm.batch_size}")
except Exception as e:
    print(f"Error: {e}")
    print("Manteniendo batch_size=32 por defecto.")
    
# Limpiamos el modelo de estrés
del modelo_stress, modelo_lig_stress, trainer_finder, tuner
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Buscando el Batch Size máximo...


`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 32 succeeded, trying batch size 64
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 64 succeeded, trying batch size 128
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 128 succeeded, trying batch size 256
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 256 succeeded, trying batch size 512
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 512 succeeded, trying batch size 1024
Batch size 1024 failed, trying batch size 512
Finished batch size finder, will continue with full run using batch size 512
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.scale_batch_size_492394b4-8b21-4ace-bbc5-898011b2aa8a.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.scale_batch_size_492394b4-8b21-4ace-bbc5-898011b2aa8a.ckpt


Batch size bruto encontrado: 512. Batch size aplicado: 64


# 5. Entrenamiento de Modelos
Parámetros globales para el entrenamiento.


In [22]:
max_epochs = 100
callbacks = [EarlyStopping(monitor="val_loss", mode="min", patience=5)]

## Modelo 1: NET-1 (Regresión Logística)


In [23]:
cleanup()
net1 = nn.Sequential(nn.Linear(224 * 224, 1))
modelo_net1 = RedNeuronal(net1, is_cnn=False, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_net1 = CSVLogger("logs", name="net1")
trainer_net1 = L.Trainer(max_epochs=max_epochs, logger=logger_net1, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])

# LR Finder
Tuner(trainer_net1).lr_find(modelo_net1, datamodule=dm)
print(f"Mejor LR sugerida para NET-1: {modelo_net1.lr}")

# Train & Test
trainer_net1.fit(model=modelo_net1, datamodule=dm)
trainer_net1.test(model=modelo_net1, datamodule=dm)

# Destruir modelo para liberar RAM/VRAM
del net1, modelo_net1, trainer_net1
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

LR finder stopped early after 70 steps due to diverging loss.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_f58a207d-b2a3-480d-99e7-fb58a5365479.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_f58a207d-b2a3-480d-99e7-fb58a5365479.ckpt
Learning rate set to 1.9054607179632464e-05
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Mejor LR sugerida para NET-1: 1.9054607179632464e-05


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │ 50.2 K │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 50.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 50.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8208333253860474     │
│         test_loss         │    0.40803274512290955    │
└───────────────────────────┴───────────────────────────┘

## Modelo 2: NET-2 (Perceptrón Multicapa)


In [24]:
cleanup()
net2 = nn.Sequential(nn.Linear(224 * 224, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
modelo_net2 = RedNeuronal(net2, is_cnn=False, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_net2 = CSVLogger("logs", name="net2")
trainer_net2 = L.Trainer(max_epochs=max_epochs, logger=logger_net2, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])

Tuner(trainer_net2).lr_find(modelo_net2, datamodule=dm)
trainer_net2.fit(model=modelo_net2, datamodule=dm)
trainer_net2.test(model=modelo_net2, datamodule=dm)

del net2, modelo_net2, trainer_net2
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

LR finder stopped early after 83 steps due to diverging loss.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_ab941a38-5035-4400-b5e2-014659014c42.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_ab941a38-5035-4400-b5e2-014659014c42.ckpt
Learning rate set to 5.7543993733715664e-05
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │  6.4 M │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 6.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.4 M                                                                                                
Total estimated model params size (MB): 25                                                                         
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8166666626930237     │
│         test_loss         │    0.4296739995479584     │
└───────────────────────────┴───────────────────────────┘

## Modelo 3: CNN Simple (Base)


In [25]:
cleanup()
cnn_base = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 56 * 56, 16), nn.ReLU(), nn.Linear(16, 1)
)
modelo_cnn = RedNeuronal(cnn_base, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn = CSVLogger("logs", name="cnn_base")
trainer_cnn = L.Trainer(max_epochs=max_epochs, logger=logger_cnn, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])

Tuner(trainer_cnn).lr_find(modelo_cnn, datamodule=dm)
trainer_cnn.fit(model=modelo_cnn, datamodule=dm)
trainer_cnn.test(model=modelo_cnn, datamodule=dm)

del cnn_base, modelo_cnn, trainer_cnn
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

LR finder stopped early after 89 steps due to diverging loss.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_3df80b4e-3a45-434a-b066-09ba748ed14f.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_3df80b4e-3a45-434a-b066-09ba748ed14f.ckpt
Learning rate set to 5.7543993733715664e-05
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │  1.6 M │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=100` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │           0.875           │
│         test_loss         │    0.2785336673259735     │
└───────────────────────────┴───────────────────────────┘

## Modelo 4: CNN Variante A (BatchNorm + Dropout)


In [26]:
cleanup()
cnn_a = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 56 * 56, 16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 1)
)
modelo_cnn_a = RedNeuronal(cnn_a, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn_a = CSVLogger("logs", name="cnn_var_a")
trainer_cnn_a = L.Trainer(max_epochs=max_epochs, logger=logger_cnn_a, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])

Tuner(trainer_cnn_a).lr_find(modelo_cnn_a, datamodule=dm)
trainer_cnn_a.fit(model=modelo_cnn_a, datamodule=dm)
trainer_cnn_a.test(model=modelo_cnn_a, datamodule=dm)

del cnn_a, modelo_cnn_a, trainer_cnn_a
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_dad49c15-7425-490d-9c57-f0d6c88ffb29.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_dad49c15-7425-490d-9c57-f0d6c88ffb29.ckpt
Learning rate set to 9.120108393559096e-06
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │  1.6 M │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 17                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8958333134651184     │
│         test_loss         │    0.2915322482585907     │
└───────────────────────────┴───────────────────────────┘

## Modelo 5: CNN Variante B (Profunda con Dropout)


In [27]:
cleanup()
cnn_b = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 28 * 28, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 1)
)
modelo_cnn_b = RedNeuronal(cnn_b, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn_b = CSVLogger("logs", name="cnn_var_b")
trainer_cnn_b = L.Trainer(max_epochs=max_epochs, logger=logger_cnn_b, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])

Tuner(trainer_cnn_b).lr_find(modelo_cnn_b, datamodule=dm)
trainer_cnn_b.fit(model=modelo_cnn_b, datamodule=dm)
trainer_cnn_b.test(model=modelo_cnn_b, datamodule=dm)

del cnn_b, modelo_cnn_b, trainer_cnn_b
cleanup()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

LR finder stopped early after 99 steps due to diverging loss.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_bd1b8739-8565-43e7-96e1-b7ac1abe8119.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_bd1b8739-8565-43e7-96e1-b7ac1abe8119.ckpt
Learning rate set to 0.001584893192461114
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │  3.2 M │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 3.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.2 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 18                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.9291666746139526     │
│         test_loss         │    0.16593757271766663    │
└───────────────────────────┴───────────────────────────┘

# Fase 3: Análisis de Pesos y Mapas de Características

## 1. Carga del Modelo Base (CNN)
Cargamos los pesos del entrenamiento previo (Fase 2) para el Modelo 3.

In [ ]:

cleanup()
# Re-definimos la arquitectura del Modelo 3 (Base)
cnn_base_arch = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 56 * 56, 16), nn.ReLU(), nn.Linear(16, 1)
)

# Cargamos el checkpoint (Version 2 fue la más completa)
checkpoint_path = 'logs/cnn_base/version_2/checkpoints/epoch=99-step=3000.ckpt'
modelo_visual = RedNeuronal.load_from_checkpoint(
    checkpoint_path, 
    model=cnn_base_arch,
    is_cnn=True
)
modelo_visual.eval() # Modo evaluación
print("Modelo cargado correctamente para análisis.")


## 2. Visualización de Filtros (Pesos)
Visualizamos los filtros de la primera capa convolucional para identificar patrones de bajo nivel.

In [ ]:

def plot_filters(layer_weights, title):
    filters = layer_weights.detach().cpu()
    n_filters = filters.shape[0]
    plt.figure(figsize=(12, 8))
    for i in range(n_filters):
        plt.subplot(4, 4, i + 1)
        # Normalizamos para mejor visualización
        f_min, f_max = filters[i].min(), filters[i].max()
        filter_img = (filters[i] - f_min) / (f_max - f_min)
        plt.imshow(filter_img.squeeze(), cmap='gray')
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

# Capa 1: Conv2d(1, 16, ...)
plot_filters(modelo_visual.model[0].weight, "Filtros de la Capa Convolucional 1 (16 filtros)")

# Capa 2: Conv2d(16, 32, ...) - Visualizamos solo los primeros 16 del primer canal de entrada
plot_filters(modelo_visual.model[3].weight[:, 0, :, :][:16], "Filtros de la Capa Convolucional 2 (Primeros 16 filtros, Canal 0)")


## 3. Visualización de Mapas de Características
Pasamos una imagen por la red y capturamos las activaciones de las capas convolucionales.

In [ ]:

def plot_feature_maps(feature_maps, title):
    n_maps = feature_maps.shape[1]
    plt.figure(figsize=(15, 10))
    # Mostramos hasta 16 mapas
    for i in range(min(n_maps, 16)):
        plt.subplot(4, 4, i + 1)
        plt.imshow(feature_maps[0, i].detach().cpu().numpy(), cmap='viridis')
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

# 1. Obtener una imagen de test
it = iter(dm.test_dataloader())
images, labels = next(it)
img = images[0:1].to(modelo_visual.device)
label = labels[0]

# Aplicamos normalización manual ya que dm.test_dataloader devuelve imagenes base
img = gpu_val_test_transforms(img)

plt.imshow(img.cpu().squeeze(), cmap='gray')
plt.title(f"Imagen Original (Clase: {dm.classes[label]})")
plt.axis('off')
plt.show()

# 2. Pasar por la primera capa
with torch.no_grad():
    x1 = modelo_visual.model[0](img) # Conv1
    plot_feature_maps(x1, "Mapas de Características - Salida Conv1")
    
    x1_pool = modelo_visual.model[2](modelo_visual.model[1](x1)) # ReLU + Pool1
    plot_feature_maps(x1_pool, "Mapas de Características - Salida Pool1")

    # 3. Pasar por la segunda capa
    x2 = modelo_visual.model[3](x1_pool) # Conv2
    plot_feature_maps(x2, "Mapas de Características - Salida Conv2")
